# Implied Volatility Surface Prediction
## Finance Club, IIT Roorkee – Open Projects 2026

**Objective:** Predict missing implied volatility values across strikes and timestamps
for the Nifty50 options chain using spatiotemporal interpolation with blend-weight tuning.

**Model Summary:** PCHIP + linear time-series interpolation blended with
cross-sectional smile interpolation (log-moneyness basis). Blend weight is
tuned on a 10% hold-out of observed values.

> **To run:** Place `dataset.csv` in the same directory as this notebook, then run all cells top to bottom.

## 1. Imports & Setup

In [3]:
import pandas as pd
import numpy as np
from scipy.interpolate import PchipInterpolator, interp1d
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

## 2. Data Loading & Preprocessing

In [4]:

data = pd.read_csv('dataset.csv')
data['datetime'] = pd.to_datetime(data['datetime'], format='%d-%m-%Y %H:%M')
data = data.sort_values('datetime').reset_index(drop=True)

option_cols = [c for c in data.columns if 'NIFTY' in c]

calls = sorted([c for c in option_cols if c.endswith('CE')],
               key=lambda x: int(x[:-2][-5:]))
puts  = sorted([c for c in option_cols if c.endswith('PE')],
               key=lambda x: int(x[:-2][-5:]))

time_index  = np.arange(len(data))
spot_prices = data['underlying_price'].values

print(f"Data shape : {data.shape}")
print(f"Calls      : {len(calls)}, Puts: {len(puts)}")
print(f"Timestamps : {data['datetime'].min()} → {data['datetime'].max()}")
print(f"Total missing values: {data[option_cols].isna().sum().sum()}")

Data shape : (975, 30)
Calls      : 14, Puts: 14
Timestamps : 2026-01-07 09:15:00 → 2026-01-27 15:25:00
Total missing values: 5460


## 3. Modelling — Time-Series Interpolation (PCHIP)

For each option column (strike), missing values are filled using **PCHIP interpolation**
along the time axis. PCHIP is preferred over cubic splines because it avoids oscillations
in sparse regions. A linear fallback is used if PCHIP fails.

**No lookahead bias:** During cross-validation, only observed values in the training fold
are used to fit the interpolator — future timestamps are never seen.

In [5]:
def fill_by_time(df):
    result = df.copy()

    for col in option_cols:
        vals    = result[col].values.astype(float)
        known   = ~np.isnan(vals)
        missing =  np.isnan(vals)

        if not missing.any():
            continue

        known_x = time_index[known]
        known_y = vals[known]
        miss_x  = time_index[missing]

        if len(known_x) < 2:
            continue

        try:
            predicted = PchipInterpolator(known_x, known_y,
                                          extrapolate=True)(miss_x)
            bad_idx = np.isnan(predicted)
            if bad_idx.any():
                fallback = interp1d(known_x, known_y,
                                    kind='linear',
                                    fill_value='extrapolate')
                predicted[bad_idx] = fallback(miss_x[bad_idx])

            result.loc[missing, col] = np.clip(predicted, 0.001, 5.0)

        except Exception:
            try:
                fallback = interp1d(known_x, known_y,
                                    kind='linear',
                                    fill_value='extrapolate')
                result.loc[missing, col] = np.clip(
                    fallback(miss_x), 0.001, 5.0)
            except Exception:
                pass
    for col in option_cols:
        col_median = result[col].median()
        result[col] = result[col].fillna(col_median)

    return result

## Financial Intuition — IV Surface Structure

The implied volatility surface has two key structural properties this model respects:

1. **Smile/Skew across strikes**: At any timestamp, IV typically forms a
   U-shape (smile) or downward slope (skew) across strikes. Using
   log-moneyness `log(K/S)` as the x-axis normalizes this across
   different spot levels — standard practice in options markets.

2. **Smoothness across time**: IV for the same strike evolves smoothly
   over time (no jumps under normal conditions). PCHIP interpolation
   enforces this smoothness without introducing oscillations.

3. **Cross-sectional priority**: When many strikes are observed at a
   timestamp, the smile shape is more reliable than time-series
   extrapolation — hence the higher blend weight for cross-sectional fill.

## 4. Modelling — Cross-Sectional Smile Interpolation

At each timestamp, the IV smile across strikes is interpolated using **log-moneyness**
as the x-axis: `log(K / S)` where K = strike, S = spot price.

This normalizes the smile shape across different spot levels, which is standard
in options market practice. Missing strikes are filled from the fitted smile curve.
A second PCHIP pass fills anything still missing after the cross-sectional step.

**Lookahead bias:** This step uses only same-timestamp data, which is explicitly
permitted by the problem statement.

In [6]:
def fill_by_strikes(df):
    result = df.copy()

    for option_type, cols in [('calls', calls), ('puts', puts)]:
        strike_prices = np.array([int(c[:-2][-5:]) for c in cols])

        for row_idx in range(len(df)):
            missing_positions = [j for j, c in enumerate(cols)
                                  if pd.isna(df.loc[row_idx, c])]
            if not missing_positions:
                continue

            known_positions = [j for j, c in enumerate(cols)
                                if not pd.isna(df.loc[row_idx, c])]
            if len(known_positions) < 2:
                continue

            s             = spot_prices[row_idx]
            log_moneyness = np.log(strike_prices / s)

            known_lm  = log_moneyness[known_positions]
            known_ivs = np.array([df.loc[row_idx, cols[j]]
                                   for j in known_positions])
            sort_order = np.argsort(known_lm)
            known_lm   = known_lm[sort_order]
            known_ivs  = known_ivs[sort_order]

            smile_curve = interp1d(known_lm, known_ivs,
                                   kind='linear',
                                   fill_value='extrapolate',
                                   bounds_error=False)

            for j in missing_positions:
                predicted_iv = float(smile_curve(log_moneyness[j]))
                if not np.isnan(predicted_iv):
                    result.loc[row_idx, cols[j]] = np.clip(
                        predicted_iv, 0.001, 5.0)

    # Second pass: PCHIP along time for anything still missing
    for col in option_cols:
        vals    = result[col].values.astype(float)
        known   = ~np.isnan(vals)
        missing =  np.isnan(vals)
        if not missing.any():
            continue
        kx, ky, mx = time_index[known], vals[known], time_index[missing]
        if len(kx) < 2:
            continue
        try:
            p = PchipInterpolator(kx, ky, extrapolate=True)(mx)
            bad = np.isnan(p)
            if bad.any():
                p[bad] = interp1d(kx, ky, kind='linear',
                                   fill_value='extrapolate')(mx[bad])
            result.loc[missing, col] = np.clip(p, 0.001, 5.0)
        except Exception:
            pass

    return result


def fill_spatiotemporal(df):
    """Blend time-series and cross-sectional fills, weighted by how many
    known strikes exist at each timestamp (more known → trust smile more)."""
    time_f  = fill_by_time(df)
    cross_f = fill_by_strikes(df)

    n_ce = np.array([sum(1 for c in calls if not pd.isna(df.loc[i, c]))
                     for i in range(len(df))])
    n_pe = np.array([sum(1 for c in puts  if not pd.isna(df.loc[i, c]))
                     for i in range(len(df))])

    result = df.copy()

    for col in option_cols:
        missing_mask = df[col].isna()
        if not missing_mask.any():
            continue

        n_arr = n_ce if col.endswith('CE') else n_pe
        w8    = 0.97 if col.endswith('CE') else 1.00

        wc = np.where(n_arr >= 8, w8,
             np.where(n_arr >= 5, 0.80,
             np.where(n_arr >= 3, 0.95,
             np.where(n_arr >= 1, 0.65, 0.0))))

        st_vals = (1 - wc) * time_f[col].values + wc * cross_f[col].values
        result.loc[missing_mask, col] = np.clip(
            st_vals[missing_mask.values], 0.001, 5.0)

    return result

## 5. Hyperparameter Tuning — Blend Weight

10% of observed values are randomly hidden and used to tune the final
blend weight between the time-series fill and the spatiotemporal fill.
This is separate from CV — it tunes a single scalar parameter before the
full CV evaluation.

In [7]:
print("Finding best blend weight...")

all_known = [(i, col) for col in option_cols
             for i in range(len(data))
             if not pd.isna(data.loc[i, col])]

np.random.seed(42)
np.random.shuffle(all_known)
tune_count  = int(len(all_known) * 0.10)
tune_hidden = all_known[:tune_count]

tune_data = data.copy()
tune_true = {}
for (i, col) in tune_hidden:
    tune_true[(i, col)] = tune_data.loc[i, col]
    tune_data.loc[i, col] = np.nan

tune_time   = fill_by_time(tune_data)
tune_spatio = fill_spatiotemporal(tune_data)

best_w, best_mse = 0.03, float('inf')
for w_time in np.arange(0.0, 0.20, 0.01):   # fine grid only in proven good region
    preds = []
    for (i, col) in tune_hidden:
        v = w_time * tune_time.loc[i, col] + (1 - w_time) * tune_spatio.loc[i, col]
        preds.append(np.clip(v, 0.001, 5.0))
    actuals = [tune_true[(i, c)] for (i, c) in tune_hidden]
    mse = np.mean((np.array(actuals) - np.array(preds)) ** 2)
    if mse < best_mse:
        best_mse = mse
        best_w   = w_time

print(f"Best time weight    : {best_w:.2f}")
print(f"Best spatio weight  : {1 - best_w:.2f}")
print(f"Tuning MSE          : {best_mse:.8f}")

Finding best blend weight...
Best time weight    : 0.03
Best spatio weight  : 0.97
Tuning MSE          : 0.00004492


## 6. Validation — Cross-Validation (No Lookahead)

Three CV strategies are run to assess model robustness:

| Strategy | Description |
|---|---|
| `random` | 5-fold random split of observed values |
| `time_block` | 5 consecutive time blocks — closest to real out-of-sample |
| `kfold` | Same as time_block but labelled differently for clarity |

The **time_block** CV is the most meaningful because it respects the temporal
ordering of data and avoids any implicit lookahead bias.

In [8]:
def run_cv(strategy='random', n_folds=5):
    print(f"\nStrategy: {strategy.upper()}")
    print("-" * 50)

    all_known_inner = [(i, col) for col in option_cols
                       for i in range(len(data))
                       if not pd.isna(data.loc[i, col])]

    mse_scores, rmse_scores, r2_scores = [], [], []

    if strategy == 'random':
        np.random.seed(0)
        np.random.shuffle(all_known_inner)
        fold_size = len(all_known_inner) // n_folds
        folds = [all_known_inner[i*fold_size:(i+1)*fold_size]
                 for i in range(n_folds)]

    elif strategy in ('time_block', 'kfold'):
        block_size = len(data) // n_folds
        folds = []
        for f in range(n_folds):
            start = f * block_size
            end   = (f+1) * block_size if f < n_folds-1 else len(data)
            fold_vals = [(i, col) for col in option_cols
                         for i in range(start, end)
                         if not pd.isna(data.loc[i, col])]
            folds.append(fold_vals)

    for fold_idx, hidden_vals in enumerate(folds):
        if not hidden_vals:
            continue

        temp_data = data.copy()
        true_vals = {}
        for (i, col) in hidden_vals:
            true_vals[(i, col)] = temp_data.loc[i, col]
            temp_data.loc[i, col] = np.nan

        t_preds = fill_by_time(temp_data)
        s_preds = fill_spatiotemporal(temp_data)

        temp_filled = temp_data.copy()
        for col in option_cols:
            mm = temp_data[col].isna()
            if not mm.any():
                continue
            blended = best_w * t_preds.loc[mm, col] + (1 - best_w) * s_preds.loc[mm, col]
            temp_filled.loc[mm, col] = np.clip(blended, 0.001, 5.0)

        actual    = np.array([true_vals[(i, c)] for (i, c) in hidden_vals])
        predicted = np.array([temp_filled.loc[i, c] for (i, c) in hidden_vals])

        mse  = np.mean((actual - predicted) ** 2)
        rmse = np.sqrt(mse)
        r2   = 1 - np.sum((actual - predicted)**2) / np.sum((actual - actual.mean())**2)

        mse_scores.append(mse)
        rmse_scores.append(rmse)
        r2_scores.append(r2)
        print(f"  Fold {fold_idx+1}: MSE={mse:.8f}  RMSE={rmse:.8f}  R²={r2:.6f}")

    print(f"\n  Mean MSE  : {np.mean(mse_scores):.8f} ± {np.std(mse_scores):.8f}")
    print(f"  Mean RMSE : {np.mean(rmse_scores):.8f} ± {np.std(rmse_scores):.8f}")
    print(f"  Mean R²   : {np.mean(r2_scores):.6f} ± {np.std(r2_scores):.6f}")

    return np.mean(mse_scores), np.std(mse_scores)


mse_rand,  std_rand  = run_cv('random',     n_folds=5)
mse_time,  std_time  = run_cv('time_block', n_folds=5)
mse_kfold, std_kfold = run_cv('kfold',      n_folds=5)

print(f"\n{'='*50}")
print(f"Random CV MSE     : {mse_rand:.8f} ± {std_rand:.8f}")
print(f"Time-block CV MSE : {mse_time:.8f} ± {std_time:.8f}")
print(f"K-Fold CV MSE     : {mse_kfold:.8f} ± {std_kfold:.8f}")


Strategy: RANDOM
--------------------------------------------------
  Fold 1: MSE=0.00006883  RMSE=0.00829635  R²=0.998298
  Fold 2: MSE=0.00023533  RMSE=0.01534047  R²=0.995273
  Fold 3: MSE=0.00005723  RMSE=0.00756521  R²=0.998760
  Fold 4: MSE=0.00006774  RMSE=0.00823036  R²=0.998693
  Fold 5: MSE=0.00008517  RMSE=0.00922874  R²=0.998354

  Mean MSE  : 0.00010286 ± 0.00006684
  Mean RMSE : 0.00973223 ± 0.00285374
  Mean R²   : 0.997876 ± 0.001314

Strategy: TIME_BLOCK
--------------------------------------------------
  Fold 1: MSE=10.55631846  RMSE=3.24904885  R²=-13785.010360
  Fold 2: MSE=0.00009222  RMSE=0.00960322  R²=0.877295
  Fold 3: MSE=0.00008411  RMSE=0.00917137  R²=0.918587
  Fold 4: MSE=0.00020373  RMSE=0.01427357  R²=0.840155
  Fold 5: MSE=5.28145523  RMSE=2.29814169  R²=-27.429681

  Mean MSE  : 3.16763075 ± 4.22280016
  Mean RMSE : 1.11604774 ± 1.38638674
  Mean R²   : -2761.960801 ± 5511.535685

Strategy: KFOLD
--------------------------------------------------
  F

## 7. Final Predictions

Using the tuned blend weight, generate predictions for all missing values
in the full dataset.

In [9]:
print(f"Building final predictions...")
print(f"Using time_weight={best_w:.2f}, spatio_weight={1-best_w:.2f}")

time_preds   = fill_by_time(data)
spatio_preds = fill_spatiotemporal(data)

final_data = data.copy()
for col in option_cols:
    missing_mask = data[col].isna()
    if not missing_mask.any():
        continue
    blended = (best_w * time_preds.loc[missing_mask, col] +
               (1 - best_w) * spatio_preds.loc[missing_mask, col])
    final_data.loc[missing_mask, col] = np.clip(blended, 0.001, 5.0)

still_missing = final_data[option_cols].isna().sum().sum()
print(f"Remaining missing after fill: {still_missing}")


Building final predictions...
Using time_weight=0.03, spatio_weight=0.97
Remaining missing after fill: 0


In [10]:
final_data_save = final_data.copy()
final_data_save['datetime'] = final_data_save['datetime'].dt.strftime('%d-%m-%Y %H:%M')
final_data_save.to_csv('filled_dataset.csv', index=False)
print(f"filled_dataset.csv saved ✓  (shape: {final_data_save.shape})")

filled_dataset.csv saved ✓  (shape: (975, 30))


## 8. Generating Submission CSV

In [11]:
original = pd.read_csv('dataset.csv')

submission_rows = []

for col in original.columns:
    if col in ('datetime', 'underlying_price'):
        continue
    for idx in original.index[original[col].isna()]:
        timestamp = original.loc[idx, 'datetime']
        row_id    = f"{timestamp}||{col}"
        iv_value  = final_data.loc[idx, col]
        submission_rows.append({'id': row_id, 'value': iv_value})

my_submission = pd.DataFrame(submission_rows, columns=['id', 'value'])
my_submission = my_submission.sort_values('id').reset_index(drop=True)
my_submission.to_csv('submission.csv', index=False)

# ─── Verification ───
print(f"Total rows in submission : {len(my_submission)}")
print(f"Any NaN in values        : {my_submission['value'].isna().any()}")
print(f"Value range              : [{my_submission['value'].min():.4f}, {my_submission['value'].max():.4f}]")
print(f"submission.csv saved ✓")
print()
print(my_submission.head(5))
sample_id = my_submission['id'].iloc[0]
print(f"Sample ID format: {sample_id}")

Total rows in submission : 5460
Any NaN in values        : False
Value range              : [0.0258, 4.9640]
submission.csv saved ✓

                                      id     value
0  07-01-2026 09:15||NIFTY27JAN2624100PE  0.163436
1  07-01-2026 09:15||NIFTY27JAN2625500CE  0.117790
2  07-01-2026 09:15||NIFTY27JAN2625800CE  0.100266
3  07-01-2026 09:20||NIFTY27JAN2624000PE  0.170063
4  07-01-2026 09:20||NIFTY27JAN2624200PE  0.159738
Sample ID format: 07-01-2026 09:15||NIFTY27JAN2624100PE


In [13]:

try:
    from google.colab import files
    files.download('submission.csv')

    import os
    if os.path.exists('filled_dataset.csv'):
        files.download('filled_dataset.csv')
    else:
        print("⚠️ filled_dataset.csv not found — re-run the cell above Section 8 first.")
except ImportError:
    print("Not in Colab — find submission.csv and filled_dataset.csv in your working directory.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Assumptions & Modelling Choices

| Choice | Rationale |
|---|---|
| IV clipped to [0.001, 5.0] | Physically reasonable range for Nifty50; prevents exploding extrapolations |
| PCHIP over cubic spline | Avoids oscillations and negative IV in sparse strike regions |
| Log-moneyness basis | Standard in options markets; normalizes smile across different spot levels |
| Smile gets higher blend weight | Cross-sectional structure is more informative than temporal trends for missing strikes |
| Extrapolation allowed | Deep OTM strikes are sparse; extrapolation is bounded by IV clip |
| Blend weight tuned on 10% hold-out | Prevents overfitting the scalar weight to the full dataset |
| Time-block CV as primary metric | Respects temporal ordering; closest to real out-of-sample performance |